In [1]:
from massspecgym.data.data_module import MassSpecDataModule
from massspecgym.data.datasets import RetrievalDataset, MassSpecDataset
from chemembed_transforms import ChemEmbedSpecTransform, ChemEmbedMolTransform
from pathlib import Path
from chemembed_model import ChemEmbedRetrieval
from pytorch_lightning import Trainer
from massspecgym.models.base import Stage
import pandas as pd


c:\Users\aa\anaconda3\envs\massspecgym\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Failed to find the pandas get_adjustment() function to patch
Failed to patch pandas - PandasTools will have limited functionality


## Training

In [ ]:
# Train dataset, without reference candidates
train_dataset = MassSpecDataset(
    spec_transform=ChemEmbedSpecTransform(),
    mol_transform=ChemEmbedMolTransform(model_path='Mol2Vec_model/model_300dim.pkl'),
)
data_module_train = MassSpecDataModule(dataset=train_dataset, batch_size=32)

data_module_train.prepare_data()  # Explicit call needed for validate before fit
data_module_train.setup()  # Explicit call needed for validate before fit

# Init model
model = ChemEmbedRetrieval(
    log_only_loss_at_stages=[Stage.TRAIN, Stage.VAL]
)

# Init trainer
trainer = Trainer(
    accelerator="auto", # Automatically use GPU if available
    max_epochs=15,        
    log_every_n_steps=1
)

# Validation before training
trainer.validate(model, datamodule=data_module_train)

# Train
trainer.fit(model, datamodule=data_module_train)

# Save model checkpoint
trainer.save_checkpoint("chemembed_trained_on_massspecgym.ckpt")


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
c:\Users\aa\anaconda3\envs\massspecgym\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:441: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Validation DataLoader 0: 100%|██████████| 5/5 [00:13<00:00,  0.37it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     Validate metric           DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        val_loss            132.37408447265625
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────



  | Name | Type      | Params
-----------------------------------
0 | cnn  | CNN_Class | 282 M 
-----------------------------------
282 M     Trainable params
0         Non-trainable params
282 M     Total params
1,129.554 Total estimated model params size (MB)


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\aa\anaconda3\envs\massspecgym\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:441: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


c:\Users\aa\anaconda3\envs\massspecgym\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:441: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Epoch 0: 100%|██████████| 5/5 [01:12<00:00,  0.07it/s, v_num=4, train_loss=61.30]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|██████████| 5/5 [00:56<00:00,  0.09it/s, v_num=4, train_loss=20.10, val_loss=123.0]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|██████████| 5/5 [01:09<00:00,  0.07it/s, v_num=4, train_loss=28.90, val_loss=93.90]

`Trainer.fit` stopped: `max_epochs=2` reached.


Epoch 1: 100%|██████████| 5/5 [01:26<00:00,  0.06it/s, v_num=4, train_loss=28.90, val_loss=93.90]


## Test

In [7]:

model = ChemEmbedRetrieval.load_from_checkpoint("Outputs/ChemEmbed original training/chemembed_trained_on_massspecgym.ckpt")

# Init trainer
trainer = Trainer(
    accelerator="auto", # Automatically use GPU if available
    max_epochs=15,        
    log_every_n_steps=1
)

# Test dataset with reference candidates
test_dataset = RetrievalDataset(
    spec_transform=ChemEmbedSpecTransform(),
    mol_transform=ChemEmbedMolTransform(model_path='Mol2Vec_model/model_300dim.pkl'),
)
data_module_test = MassSpecDataModule(dataset=test_dataset, batch_size=32)
data_module_test.setup(stage="test")

# Test
test_results = trainer.test(model, datamodule=data_module_test)

# Save test results to CSV
results_df = pd.DataFrame(test_results)
results_df.to_csv("masspecgym_chemembed_results.csv", index=False)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
c:\Users\aa\anaconda3\envs\massspecgym\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:441: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Testing DataLoader 0:   2%|▏         | 10/549 [13:36<12:13:23,  0.01it/s]

c:\Users\aa\anaconda3\envs\massspecgym\Lib\site-packages\pytorch_lightning\trainer\call.py:54: Detected KeyboardInterrupt, attempting graceful shutdown...
